<a href="https://colab.research.google.com/github/AbenezerYBekele/AI-For-Beginners/blob/main/spacex_launch_site_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpaceX Launch Site Location Analysis

**Interactive geospatial analysis of SpaceX launch sites, built with Folium.**

This project visualizes where SpaceX launches its rockets from, how successful
each launch site has been, and how each site's location relates to nearby
geographic features (coastlines, highways, railways, and cities).

**Contents**
1. Data Preparation
2. Map All Launch Sites
3. Visualize Launch Outcomes by Site
4. Proximity Analysis
5. Key Findings


## 1. Setup & Data Preparation

In [ ]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import radians, sin, cos, sqrt, atan2

pd.set_option("display.max_columns", None)


In [ ]:
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
)

# Load launch records: one row per historical SpaceX launch, with the
# launch site name, its coordinates, and the mission outcome ("class":
# 1 = successful landing, 0 = failed landing).
spacex_df = pd.read_csv(DATA_URL)[["Launch Site", "Lat", "Long", "class"]]

# Collapse to one row per unique launch site, for plotting site locations.
launch_sites_df = (
    spacex_df.groupby("Launch Site", as_index=False)
    .first()[["Launch Site", "Lat", "Long"]]
)

launch_sites_df


,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


## 2. Map All Launch Sites

Each launch site is marked with a circle (click for its name) and a
permanent text label, centered on NASA's Johnson Space Center in Houston, TX.


In [ ]:
NASA_JSC_COORDINATE = [29.559684888503615, -95.0830971930759]


def build_base_map(center, zoom_start=5):
    """Create a Folium map centered on the given [lat, lon] coordinate."""
    return folium.Map(location=center, zoom_start=zoom_start)


def add_launch_site_markers(site_map, sites_df):
    """
    Draw a circle and a text label at each launch site's coordinates.

    Parameters
    ----------
    site_map : folium.Map
        Map instance to draw on.
    sites_df : pandas.DataFrame
        Must contain 'Launch Site', 'Lat', and 'Long' columns.

    Returns
    -------
    folium.Map
        The same map, with markers added.
    """
    for _, site in sites_df.iterrows():
        coordinate = [site["Lat"], site["Long"]]

        circle = folium.Circle(
            coordinate,
            radius=1000,
            color="#000000",
            fill=True,
        ).add_child(folium.Popup(site["Launch Site"]))

        label = folium.map.Marker(
            coordinate,
            icon=DivIcon(
                icon_size=(20, 20),
                icon_anchor=(0, 0),
                html=(
                    '<div style="font-size:12px; color:#d35400;">'
                    f'<b>{site["Launch Site"]}</b></div>'
                ),
            ),
        )

        site_map.add_child(circle)
        site_map.add_child(label)

    return site_map


site_map = build_base_map(NASA_JSC_COORDINATE)
site_map = add_launch_site_markers(site_map, launch_sites_df)
site_map


## 3. Visualize Launch Outcomes by Site

Every individual launch is plotted and color-coded by outcome:
**green = successful landing**, **red = failed landing**. Markers are
clustered so nearby launches at the same site remain readable when zoomed out.


In [ ]:
def assign_marker_colors(df):
    """Map each launch outcome to a marker color: green if successful, else red."""
    return df["class"].apply(lambda outcome: "green" if outcome == 1 else "red")


def add_launch_outcome_markers(site_map, launches_df):
    """
    Add a color-coded, clustered marker for every individual launch.

    Parameters
    ----------
    site_map : folium.Map
        Map instance to draw on.
    launches_df : pandas.DataFrame
        Must contain 'Lat', 'Long', and 'class' columns.

    Returns
    -------
    folium.Map
        The same map, with a marker cluster of launch outcomes added.
    """
    marker_cluster = MarkerCluster()
    site_map.add_child(marker_cluster)

    launches_df = launches_df.copy()
    launches_df["marker_color"] = assign_marker_colors(launches_df)

    for _, launch in launches_df.iterrows():
        marker = folium.Marker(
            [launch["Lat"], launch["Long"]],
            icon=folium.Icon(color="white", icon_color=launch["marker_color"]),
        )
        marker_cluster.add_child(marker)

    return site_map


site_map = add_launch_outcome_markers(site_map, spacex_df)
site_map


## 4. Proximity Analysis

To understand *why* a launch site might be located where it is, we measure
its distance to nearby infrastructure and geography (coastline, highway,
railway, city). A `MousePosition` readout makes it easy to find the
coordinates of any point of interest directly on the map.


In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Compute the great-circle distance between two points on Earth.

    Parameters
    ----------
    lat1, lon1, lat2, lon2 : float
        Latitude/longitude of each point, in decimal degrees.

    Returns
    -------
    float
        Distance between the two points, in kilometers.
    """
    earth_radius_km = 6373.0

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    d_lat = lat2 - lat1
    d_lon = lon2 - lon1

    a = sin(d_lat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(d_lon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return earth_radius_km * c


def add_proximity_line(site_map, launch_site_coord, poi_coord, poi_label="POI"):
    """
    Mark a point of interest, label it with its distance from a launch
    site, and draw a connecting line between the two.

    Parameters
    ----------
    site_map : folium.Map
        Map instance to draw on.
    launch_site_coord : list[float]
        [lat, lon] of the launch site.
    poi_coord : list[float]
        [lat, lon] of the point of interest (coastline, highway, etc.).
    poi_label : str
        Descriptive name used only for the return value / bookkeeping.

    Returns
    -------
    tuple[folium.Map, float]
        The updated map and the computed distance in kilometers.
    """
    distance_km = haversine_distance(*launch_site_coord, *poi_coord)

    distance_marker = folium.Marker(
        poi_coord,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=(
                '<div style="font-size:12px; color:#d35400;">'
                f'<b>{distance_km:.2f} KM</b></div>'
            ),
        ),
    )
    site_map.add_child(distance_marker)
    site_map.add_child(folium.PolyLine(locations=[launch_site_coord, poi_coord], weight=1))

    return site_map, distance_km


mouse_position = MousePosition(
    position="topright",
    separator=" Long: ",
    empty_string="NaN",
    lng_first=False,
    num_digits=5,
    prefix="Lat:",
)
site_map.add_child(mouse_position)
site_map


In [ ]:
ccafs_slc40_coord = [28.563197, -80.576820]
nearest_coastline_coord = [28.56367, -80.57163]

site_map, coastline_distance_km = add_proximity_line(
    site_map, ccafs_slc40_coord, nearest_coastline_coord, poi_label="Coastline"
)

print(f"CCAFS SLC-40 -> nearest coastline point: {coastline_distance_km:.2f} km")
site_map


CCAFS SLC-40 -> nearest coastline point: 0.51 km


## 5. Key Findings

- **Coastal proximity:** Every launch site sits close to a coastline. This
  lets rockets fly out over open ocean, so any failure during ascent falls
  over water rather than populated land.
- **Transport access:** Sites are close enough to highways and railways to
  move rocket stages and ground equipment efficiently.
- **Distance from cities:** Despite being near infrastructure, launch sites
  keep a clear buffer from cities and dense population centers, for safety.
- **Success rate by site:** The clustered map in Section 3 makes it easy to
  compare green/red marker density across sites, highlighting which
  locations have historically had the highest landing success rates.
